In [0]:
# Databricks notebook source
import json
from datetime import datetime

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("table_metadata", "")

run_id = dbutils.widgets.get("run_id").strip()
table_name = dbutils.widgets.get("table_name").strip()
table_metadata_raw = dbutils.widgets.get("table_metadata").strip()

def resolve_gold_table_metadata(table_name_value, table_metadata_value):
    if table_metadata_value:
        return json.loads(table_metadata_value.replace("'", '"'))

    if not table_name_value:
        return None

    safe_table_name = table_name_value.replace("'", "''")
    rows = spark.sql(f"""
        SELECT *
        FROM banking.metadata.tables
        WHERE lower(target_layer) = 'gold'
          AND lower(table_name) = lower('{safe_table_name}')
        LIMIT 2
    """).collect()

    if not rows:
        raise ValueError(f"No gold metadata found for table_name={table_name_value}")

    row = rows[0]
    return {
        "table_id": str(row.table_id) if row.table_id is not None else "",
        "table_name": str(row.table_name) if row.table_name is not None else "",
        "source_system": str(row.source_system) if row.source_system is not None else "",
        "source_schema": str(row.source_schema) if row.source_schema is not None else "",
        "source_table": str(row.source_table) if row.source_table is not None else "",
        "source_path": str(row.source_path) if row.source_path is not None else "",
        "target_layer": str(row.target_layer) if row.target_layer is not None else "",
        "bronze_schema": str(row.bronze_schema) if row.bronze_schema is not None else "",
        "silver_schema": str(row.silver_schema) if row.silver_schema is not None else "",
        "gold_schema": str(row.gold_schema) if row.gold_schema is not None else "",
        "active_flag": str(row.active_flag) if row.active_flag is not None else "",
        "load_order": str(row.load_order) if row.load_order is not None else "",
        "created_at": str(row.created_at) if row.created_at is not None else ""
    }

table_metadata = resolve_gold_table_metadata(table_name, table_metadata_raw)

if not run_id or not table_metadata:
    print("Widgets are ready.")
    print("Set run_id and either table_name or table_metadata, then rerun the cell.")
    print("Example table_name: branch_performance")
else:
    table_id = table_metadata["table_id"]
    table_name = table_metadata["table_name"]
    dbutils.widgets.text("table_metadata", json.dumps(table_metadata))

    start_time = datetime.utcnow()

    print("Run ID:", run_id)
    print("Table ID:", table_id)
    print("Table Name:", table_name)
    print("Resolved table_metadata widget from inputs.")


In [0]:
# COMMAND ----------

import json
from datetime import datetime

spark.sql("CREATE SCHEMA IF NOT EXISTS banking.gold")

run_id = dbutils.widgets.get("run_id").strip()
table_metadata = dbutils.widgets.get("table_metadata").strip()

if not run_id or not table_metadata:
    print("Set run_id and table_metadata, then rerun this cell.")
else:
    table_metadata = json.loads(table_metadata.replace("'", '"'))
    table_id = table_metadata["table_id"]
    table_name = table_metadata["table_name"]
    start_time = datetime.utcnow()

    # COMMAND ----------

    entry_exists = spark.sql(f"""
        SELECT 1
        FROM banking.metadata.pipeline_runs
        WHERE run_id = {run_id}
        AND table_id = {table_id}
    """).count() > 0

    if entry_exists:
        spark.sql(f"""
            UPDATE banking.metadata.pipeline_runs
            SET
                layer = 'Gold',
                start_time = TIMESTAMP('{start_time}'),
                end_time = NULL,
                status = 'INPROGRESS',
                number_of_records = NULL,
                error_message = NULL
            WHERE run_id = {run_id}
            AND table_id = {table_id}
        """)
    else:
        spark.sql(f"""
            INSERT INTO banking.metadata.pipeline_runs
            VALUES (
                {run_id},
                {table_id},
                'Gold',
                TIMESTAMP('{start_time}'),
                NULL,
                'INPROGRESS',
                NULL,
                NULL
            )
        """)

    print("Audit entry created / updated")

    # COMMAND ----------

    notebook_path = f"gold_transformations/{table_name}"

    print("Notebook to execute:", notebook_path)

    # COMMAND ----------

    status = "SUCCESS"
    error_message = None
    records = None

    try:
        result = dbutils.notebook.run(
            notebook_path,
            timeout_seconds=0
        )

        if result:
            records = int(result)

        print("Notebook completed successfully")
        print("Records:", records)

    except Exception as e:
        status = "FAILED"
        error_message = str(e)

        print("Notebook failed")
        print(error_message)

    # COMMAND ----------

    end_time = datetime.utcnow()

    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            end_time = TIMESTAMP('{end_time}'),
            status = '{status}',
            number_of_records = {records if records else 'NULL'},
            error_message = {f"'{error_message}'" if error_message else 'NULL'}
        WHERE run_id = {run_id}
        AND table_id = {table_id}
    """)

    print("Audit table updated")

    # COMMAND ----------

    if status == "FAILED":
        raise Exception(error_message)